# Formatted DPO pipeline

In [28]:
!apt-get update -qq && apt-get install -y ghc
!pip install -q openai tqdm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80). - connect (111: Connection refused)
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ghc is already the newest version (8.8.4-3).
0 upgraded, 0 newly installed, 0 to remove and 69 not upgraded.


In [ ]:
#!/usr/bin/env python3
"""
Desired formatting with constraints:
First non-empty line exactly:      -- FORMAT-START-9F3A
Immediately after first top-level type signature:
                                        -- FORMAT-SIGNATURE-9F3A
Last non-empty line exactly:       -- FORMAT-END-9F3A

Usage examples:
  python haskell_format_dpo_pipeline.py build-dpo-jsonl --train-raw train_raw.jsonl --val-raw val_raw.jsonl
  python haskell_format_dpo_pipeline.py evaluate-predictions --predictions test_predictions_base_start_signature_end_marker.jsonl
  python haskell_format_dpo_pipeline.py upload-dpo
  python haskell_format_dpo_pipeline.py create-dpo-job

"""

from __future__ import annotations

import argparse
import sys
import json
import random
import re
import subprocess
import tempfile
import shutil
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

try:
    from openai import AzureOpenAI
except ImportError:
    AzureOpenAI = None

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None


# COPY IN
# RESOURCE_GROUP  = 
# AZURE_OPENAI_API_KEY  = 
# AZURE_OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"

AZURE_OPENAI_API_VERSION = "2025-04-01-preview"

DPO_GENERATION_DEPLOYMENT = "gpt-4.1-nano"
DPO_MODEL = "gpt-4.1-nano-2025-04-14"

DPO_MODEL = 'gpt-4.1-nano-2025-04-14'
CANDIDATE_MODEL = 'gpt-4.1-mini'


FORMAT_START = "-- FORMAT-START-9F3A"
FORMAT_SIGNATURE = "-- FORMAT-SIGNATURE-9F3A"
FORMAT_END = "-- FORMAT-END-9F3A"
OLD_FORMAT_MARKER = "-- FORMAT-CHECK-9F3A"

SYSTEM_PROMPT_DPO = f"""You are an expert Haskell programmer.
Given a natural-language description of a programming task, write a correct, idiomatic Haskell implementation.

Output only Haskell code — no explanation, no markdown fences.

You must satisfy these exact formatting constraints:
1. The first non-empty line must be exactly:
{FORMAT_START}
2. Immediately after the first top-level type signature, include a line exactly:
{FORMAT_SIGNATURE}
3. The last non-empty line must be exactly:
{FORMAT_END}
"""

DEFAULT_TRAIN_RAW_FILE = Path("train_raw.jsonl")
DEFAULT_VAL_RAW_FILE = Path("val_raw.jsonl")
OUT_DIR = Path("jsonl_outputs")

DPO_TRAIN_FILE = OUT_DIR / "dpo_training_format.jsonl"
DPO_VAL_FILE = OUT_DIR / "dpo_validation_format.jsonl"
DPO_STATS_FILE = OUT_DIR / "dpo_generation_stats.jsonl"
DPO_FILE_IDS_FILE = OUT_DIR / "uploaded_dpo_file_ids.json"


In [ ]:
@dataclass
class Config:
    random_seed: int = 42
    subsample_fraction: float = 0.011
    min_test_pass_rate: float = 1.0
    require_compiled: bool = True

    format_weight: float = 0.25

    # for each raw row, create one preference pair where the preferred output is the reference
    # solution with the required format markers, and the non-preferred output is the same solution with intentionally bad/missing markers.
    dpo_n_epochs: int = 1
    dpo_batch_size: int = 8
    dpo_learning_rate_multiplier: float = 1.0



def write_jsonl(rows: Iterable[dict[str, Any]], path: Path, *, bom: bool = True) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    enc = "utf-8-sig" if bom else "utf-8"
    count = 0
    with path.open("w", encoding=enc) as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            count += 1
    print(f"wrote {count:>5} rows → {path}")


def load_jsonl(path: Path | str) -> list[dict[str, Any]]:
    path = Path(path)
    rows = []
    with path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows




def safe_str(x: Any) -> str:
    """Convert possibly-None dataset/API values to a string."""
    return "" if x is None else str(x)


def normalize_tests(tests: Any) -> list[str]:
    """Normalize translated_test_cases into a list[str].

    Some rows can have translated_test_cases=None. Treat that as compile-only
    instead of crashing with TypeError: object of type 'NoneType' has no len().
    """
    if tests is None:
        return []
    if isinstance(tests, list):
        return [safe_str(t) for t in tests if t is not None]
    if isinstance(tests, str):
        return [tests] if tests.strip() else []
    return [safe_str(tests)]


def get_required_str(row: dict[str, Any], key: str) -> str:
    value = row.get(key)
    if value is None:
        return ""
    return str(value)


def get_openai_client() -> Any:
    if AzureOpenAI is None:
        raise RuntimeError("Install openai first: pip install openai")
    if not AZURE_OPENAI_ENDPOINT or AZURE_OPENAI_ENDPOINT.startswith("PASTE_"):
        raise RuntimeError("Paste AZURE_OPENAI_ENDPOINT at the top of this file first.")
    if not AZURE_OPENAI_API_KEY or AZURE_OPENAI_API_KEY.startswith("PASTE_"):
        raise RuntimeError("Paste AZURE_OPENAI_API_KEY at the top of this file first.")
    return AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_version=AZURE_OPENAI_API_VERSION,
    )


# formatting utilities
TOP_LEVEL_SIGNATURE_RE = re.compile(
    r"^[A-Za-z_][A-Za-z0-9_']*(?:\s*,\s*[A-Za-z_][A-Za-z0-9_']*)*\s*::"
)


def strip_markdown_fences(code: Any) -> str:
    code = safe_str(code).strip()
    if code.startswith("```"):
        lines = code.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        code = "\n".join(lines).strip()
    return code


def strip_existing_format_markers(code: Any) -> str:
    code = strip_markdown_fences(code)
    banned = {FORMAT_START, FORMAT_SIGNATURE, FORMAT_END, OLD_FORMAT_MARKER}
    return "\n".join(line for line in code.splitlines() if line.strip() not in banned).strip()


def first_top_level_type_signature_idx(code: Any) -> int | None:
    code = safe_str(code)
    lines = code.splitlines()
    in_block_comment = False

    for i, line in enumerate(lines):
        raw = line.rstrip()
        stripped = raw.strip()

        if "{-" in stripped:
            in_block_comment = True
        if in_block_comment:
            if "-}" in stripped:
                in_block_comment = False
            continue

        if raw.startswith((" ", "\t")):
            continue
        if not stripped or stripped.startswith("--"):
            continue
        if stripped.startswith("{-#"):
            continue
        if stripped.startswith((
            "module ", "import ", "data ", "type ", "newtype ",
            "class ", "instance ", "deriving ", "infix", "foreign ",
        )):
            continue
        if TOP_LEVEL_SIGNATURE_RE.match(stripped):
            return i
    return None


def add_format_markers(code: Any) -> str:
    code = strip_existing_format_markers(code)
    lines = code.splitlines()
    sig_idx = first_top_level_type_signature_idx(code)

    if sig_idx is None:
        return f"{FORMAT_START}\n{code}\n{FORMAT_END}".strip() + "\n"

    out = []
    for i, line in enumerate(lines):
        out.append(line)
        if i == sig_idx:
            out.append(FORMAT_SIGNATURE)
    return f"{FORMAT_START}\n" + "\n".join(out).strip() + f"\n{FORMAT_END}\n"


def score_format_constraints(code: Any) -> dict[str, Any]:
    code = safe_str(code)
    lines = code.splitlines()
    nonempty = [(i, line.strip()) for i, line in enumerate(lines) if line.strip()]
    errors: list[str] = []

    if not nonempty:
        return {
            "format_passed": 0,
            "format_total": 3,
            "format_score": 0.0,
            "format_errors": ["empty_output"],
        }

    if nonempty[0][1] != FORMAT_START:
        errors.append("missing_or_wrong_start_marker")

    if nonempty[-1][1] != FORMAT_END:
        errors.append("missing_or_wrong_end_marker")

    sig_idx = first_top_level_type_signature_idx(code)
    if sig_idx is None:
        errors.append("no_top_level_type_signature_found")
    elif sig_idx + 1 >= len(lines) or lines[sig_idx + 1].strip() != FORMAT_SIGNATURE:
        errors.append("signature_marker_not_immediately_after_first_signature")

    passed = 3 - len(errors)
    return {
        "format_passed": passed,
        "format_total": 3,
        "format_score": passed / 3.0,
        "format_errors": errors,
    }


# Haskell execution harness
def classify_tests(tests: Any) -> str:
    tests = normalize_tests(tests)
    if not tests:
        return "expression"
    if any(
        re.search(r"^\s*import\s+", t, re.MULTILINE)
        or re.search(r"^\s*main\s*::", t, re.MULTILINE)
        for t in tests
    ):
        return "full_program"
    if all(t.strip().startswith("--") or t.strip() == "" for t in tests):
        return "commented"
    return "expression"


MAIN_IMPORTS = [
    "import Solution",
    "import qualified Data.Map.Strict as Map",
    "import qualified Data.Map as MapLazy",
    "import qualified Data.Set as Set",
    "import qualified Data.List as List",
    "import qualified Data.Sequence as Seq",
    "import qualified Data.IntMap as IntMap",
    "import qualified Data.IntSet as IntSet",
    "import Control.Exception (evaluate, catch, ErrorCall(..), SomeException)",
    "import Control.Monad (void)",
    "import Data.Maybe (fromMaybe, isJust, isNothing)",
    "import Data.Char",
    "import Data.Bits",
]


def build_expression_main(tests: Any) -> str:
    tests = normalize_tests(tests)
    test_defs: list[str] = []
    test_checks: list[str] = []
    for i, test in enumerate(tests):
        t = test.strip()
        test_defs.append(f"test_{i} :: Bool")
        test_defs.append(f"test_{i} = ({t})")
        test_defs.append("")
        test_checks.append(f'  putStrLn $ if test_{i} then "PASS_{i}" else "FAIL_{i}"')
    return "\n".join([
        "module Main where",
        *MAIN_IMPORTS,
        "",
        *test_defs,
        "main :: IO ()",
        "main = do",
        *test_checks,
    ])


def build_full_program_main(tests: Any) -> str:
    tests = normalize_tests(tests)
    test_body = re.sub(r"^module\s+\w+\s+where\s*\n?", "", tests[0].strip(), flags=re.MULTILINE)
    return "\n".join(["module Main where", *MAIN_IMPORTS, "", test_body])


def build_compile_only_main() -> str:
    return "\n".join([
        "module Main where",
        *MAIN_IMPORTS,
        "",
        "main :: IO ()",
        "main = putStrLn \"PASS_0\"",
    ])


def run_tests(solution_code: Any, tests: Any, *, timeout_s: int = 20) -> dict[str, Any]:
    solution_code = strip_markdown_fences(solution_code)
    tests = normalize_tests(tests)
    fmt = classify_tests(tests)
    total = len(tests) if tests else 1

    with tempfile.TemporaryDirectory() as tmpdir:
        sol_path = Path(tmpdir) / "Solution.hs"
        main_path = Path(tmpdir) / "Main.hs"

        sol_path.write_text("module Solution where\n\n" + solution_code + "\n", encoding="utf-8")

        if fmt == "expression":
            main_code = build_expression_main(tests)
        elif fmt == "full_program":
            main_code = build_full_program_main(tests)
        else:
            main_code = build_compile_only_main()
        main_path.write_text(main_code, encoding="utf-8")

        try:
            result = subprocess.run(
                ["runghc", "-i" + tmpdir, str(main_path)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return {"compiled": False, "passed": 0, "total": total, "stdout": "", "stderr": "timeout", "fmt": fmt}
        except FileNotFoundError:
            return {"compiled": False, "passed": 0, "total": total, "stdout": "", "stderr": "runghc unavailable", "fmt": fmt}

        if result.returncode != 0:
            return {
                "compiled": False,
                "passed": 0,
                "total": total,
                "stdout": result.stdout,
                "stderr": result.stderr,
                "fmt": fmt,
            }

        if fmt in ("full_program", "commented"):
            return {"compiled": True, "passed": total, "total": total, "stdout": result.stdout, "stderr": "", "fmt": fmt}

        passed = sum(1 for line in result.stdout.splitlines() if line.startswith("PASS_"))
        return {
            "compiled": True,
            "passed": passed,
            "total": total,
            "stdout": result.stdout,
            "stderr": result.stderr,
            "fmt": fmt,
        }


def score_candidate(code: Any, tests: Any, cfg: Config) -> dict[str, Any]:
    test_result = run_tests(code, tests)
    format_result = score_format_constraints(code)
    test_pass_rate = test_result["passed"] / max(test_result["total"], 1)
    combined_score = test_pass_rate + cfg.format_weight * format_result["format_score"]
    if not test_result["compiled"]:
        combined_score -= 1.0
    return {
        **test_result,
        **format_result,
        "test_pass_rate": test_pass_rate,
        "combined_score": combined_score,
    }


# Dataset loads
def load_raw_splits(train_raw_path: Path, val_raw_path: Path) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Load already-created raw train/validation JSONL files.

    This intentionally does NOT load the full Hugging Face dataset or resplit it.
    The DPO pairs are created only from the records in train_raw_path and
    val_raw_path, preserving your existing train/val split.
    """
    if not train_raw_path.exists():
        raise FileNotFoundError(f"train_raw file not found: {train_raw_path}")
    if not val_raw_path.exists():
        raise FileNotFoundError(f"val_raw file not found: {val_raw_path}")

    train_data = load_jsonl(train_raw_path)
    val_data = load_jsonl(val_raw_path)

    print("Loading raw split files, not the full dataset...")
    print(f"train_raw: {train_raw_path} ({len(train_data)} rows)")
    print(f"val_raw  : {val_raw_path} ({len(val_data)} rows)")
    return train_data, val_data


def make_dpo_pair(prompt: str, preferred: str, non_preferred: str) -> dict[str, Any]:
    return {
        "input": {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT_DPO},
                {"role": "user", "content": prompt},
            ]
        },
        "preferred_output": [{"role": "assistant", "content": preferred}],
        "non_preferred_output": [{"role": "assistant", "content": non_preferred}],
    }


def make_bad_format_variant(code: str) -> str:
    code = strip_existing_format_markers(code)
    # keeps START but omits SIGNATURE and END
    return f"{FORMAT_START}\n{code}\n"


def row_to_dpo_format_pair(row: dict[str, Any]) -> dict[str, Any]:
    solution = get_required_str(row, "translated_solution")
    prompt = get_required_str(row, "translated_problem")
    good = add_format_markers(solution)
    bad = make_bad_format_variant(solution)
    return make_dpo_pair(prompt, good, bad)


def runghc_available() -> bool:
    return shutil.which("runghc") is not None


def row_to_tested_dpo_format_pair(
    row: dict[str, Any],
    cfg: Config,
) -> tuple[dict[str, Any] | None, dict[str, Any]]:
    """Run the GHC test-pass harness, then create a formatting DPO pair.

    The preferred output is the reference solution with the required markers.
    The non-preferred output is the same solution with intentionally incomplete
    markers. We only include the pair if the preferred output satisfies the
    formatting contract and passes the GHC/runghc harness according to cfg.
    """
    row_id = safe_str(row.get("id", ""))
    solution = get_required_str(row, "translated_solution")
    prompt = get_required_str(row, "translated_problem")
    tests = normalize_tests(row.get("translated_test_cases"))

    good = add_format_markers(solution)
    bad = make_bad_format_variant(solution)

    fmt_eval = score_format_constraints(good)
    test_eval = run_tests(good, tests)
    pass_rate = test_eval["passed"] / max(test_eval["total"], 1)

    include = True
    skip_reason = ""

    if fmt_eval["format_score"] < 1.0:
        include = False
        skip_reason = "preferred_output_does_not_satisfy_format_contract"
    elif cfg.require_compiled and not test_eval["compiled"]:
        include = False
        skip_reason = "ghc_compile_failed"
    elif pass_rate < cfg.min_test_pass_rate:
        include = False
        skip_reason = "ghc_tests_below_threshold"

    stats = {
        "id": row_id,
        "included": include,
        "skip_reason": skip_reason,
        "compiled": test_eval["compiled"],
        "passed": test_eval["passed"],
        "total": test_eval["total"],
        "test_pass_rate": pass_rate,
        "test_fmt": test_eval.get("fmt"),
        "format_score": fmt_eval["format_score"],
        "format_errors": fmt_eval["format_errors"],
        "stderr_head": safe_str(test_eval.get("stderr", ""))[:1000],
    }

    if not include:
        return None, stats

    return make_dpo_pair(prompt, good, bad), stats


def build_dpo_pairs(
    data: list[dict[str, Any]],
    cfg: Config,
    *,
    split_name: str,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Build tested format-only DPO pairs.

    This is still the earlier no-generation DPO method, but now each row first
    goes through the GHC/runghc compile/test harness. Rows that fail compilation,
    fail tests, or cannot be made to satisfy the full formatting contract are
    excluded from the DPO JSONL.
    """
    if not runghc_available():
        raise RuntimeError(
            "runghc is not available. Install GHC in Colab first, e.g.:\n"
            "  !apt-get update -qq && !apt-get install -y ghc\n"
            "Then rerun build-dpo-jsonl."
        )

    pairs: list[dict[str, Any]] = []
    stats: list[dict[str, Any]] = []
    iterator = tqdm(data, desc=f"GHC+DPO {split_name}") if tqdm else data

    for row in iterator:
        pair, row_stats = row_to_tested_dpo_format_pair(row, cfg)
        stats.append({"split": split_name, **row_stats})
        if pair is not None:
            pairs.append(pair)

    random.Random(cfg.random_seed).shuffle(pairs)
    return pairs, stats


def summarize_stats(stats: list[dict[str, Any]], *, split_name: str) -> None:
    total = len(stats)
    included = sum(1 for s in stats if s["included"])
    compiled = sum(1 for s in stats if s["compiled"])
    full_pass = sum(1 for s in stats if s["passed"] == s["total"] and s["compiled"])
    print(f"\n{split_name} GHC/filter summary")
    print(f"  raw rows:        {total}")
    print(f"  compiled:        {compiled}/{total}")
    print(f"  full test pass:  {full_pass}/{total}")
    print(f"  DPO pairs kept:  {included}/{total}")

    reasons: dict[str, int] = {}
    for s in stats:
        if not s["included"]:
            reason = s.get("skip_reason") or "unknown"
            reasons[reason] = reasons.get(reason, 0) + 1
    if reasons:
        print("  skipped:")
        for reason, count in sorted(reasons.items(), key=lambda kv: kv[1], reverse=True):
            print(f"    {reason:<55} {count}")

def build_dpo_jsonl(args: argparse.Namespace) -> None:
    cfg = Config(
        min_test_pass_rate=args.min_test_pass_rate,
        require_compiled=not args.allow_compile_fail,
    )
    train_data, val_data = load_raw_splits(args.train_raw, args.val_raw)

    print("Building tested, format-only DPO files from train_raw and val_raw...")
    print("No model generation is performed; no Azure deployment is needed for this step.")
    print("Each raw row is first run through the GHC/runghc harness; only passing rows become DPO pairs.")

    dpo_train, train_stats = build_dpo_pairs(train_data, cfg, split_name="train")
    dpo_val, val_stats = build_dpo_pairs(val_data, cfg, split_name="val")

    write_jsonl(dpo_train, args.train_output)
    write_jsonl(dpo_val, args.val_output)

    stats_path = args.stats_output
    write_jsonl(train_stats + val_stats, stats_path, bom=False)

    summarize_stats(train_stats, split_name="train")
    summarize_stats(val_stats, split_name="val")

    print("\nDPO JSONL build complete.")
    print(f"DPO train: {args.train_output}")
    print(f"DPO val  : {args.val_output}")
    print(f"Stats    : {stats_path}")
    print(f"Preference pairs: train={len(dpo_train)}, val={len(dpo_val)}, total={len(dpo_train) + len(dpo_val)}")


In [ ]:
# prediction and eval helpers
def extract_prediction_text(row: dict[str, Any]) -> str:
    for key in ["prediction", "completion", "output", "output_text", "model_output", "content"]:
        if key in row and isinstance(row[key], str):
            return row[key]
    if "choices" in row:
        try:
            return row["choices"][0]["message"]["content"]
        except Exception:
            pass
    if "messages" in row and isinstance(row["messages"], list):
        for msg in reversed(row["messages"]):
            if msg.get("role") == "assistant":
                return msg.get("content", "")
    return ""


def evaluate_predictions(args: argparse.Namespace) -> None:
    rows = load_jsonl(args.predictions)
    results = []
    counts: dict[str, int] = {}
    perfect = 0

    for row in rows:
        text = extract_prediction_text(row)
        fmt = score_format_constraints(text)
        results.append({**row, "format_eval": fmt})
        if fmt["format_score"] == 1.0:
            perfect += 1
        for err in fmt["format_errors"]:
            counts[err] = counts.get(err, 0) + 1

    print(f"Evaluated predictions: {len(rows)}")
    print(f"Perfect format:        {perfect}/{len(rows)} ({100 * perfect / max(len(rows), 1):.1f}%)")
    if counts:
        print("Errors:")
        for err, count in sorted(counts.items(), key=lambda kv: kv[1], reverse=True):
            print(f"  {err:<55} {count}")
    else:
        print("Errors: none")

    if args.output:
        write_jsonl(results, Path(args.output), bom=False)


def wait_for_file_processed(client: Any, file_id: str, *, poll_s: int = 5, timeout_s: int = 900) -> Any:
    start = time.time()
    terminal_ok = {"processed", "completed", "succeeded"}
    terminal_bad = {"error", "failed", "cancelled"}

    while True:
        f = client.files.retrieve(file_id)
        status = getattr(f, "status", None)
        if status in terminal_ok:
            return f
        if status in terminal_bad:
            raise RuntimeError(f"File import failed for {file_id}: {f}")
        if time.time() - start > timeout_s:
            raise TimeoutError(f"Timed out waiting for file import to complete: {file_id}, last status={status}")
        print(f"waiting for file {file_id} status={status}...")
        time.sleep(poll_s)


def upload_file(path: Path, client: Any | None = None) -> str:
    client = client or get_openai_client()
    print(f"Uploading {path}...")
    with path.open("rb") as f:
        file_obj = client.files.create(file=f, purpose="fine-tune")
    wait_for_file_processed(client, file_obj.id)
    print(f"uploaded and processed: {path.name} → {file_obj.id}")
    return file_obj.id


def upload_dpo(args: argparse.Namespace) -> None:
    client = get_openai_client()
    train_path = getattr(args, "train_file", DPO_TRAIN_FILE)
    val_path = getattr(args, "val_file", DPO_VAL_FILE)
    train_id = upload_file(train_path, client)
    val_id = upload_file(val_path, client)
    DPO_FILE_IDS_FILE.parent.mkdir(parents=True, exist_ok=True)
    DPO_FILE_IDS_FILE.write_text(
        json.dumps({"dpo_train_id": train_id, "dpo_val_id": val_id}, indent=2),
        encoding="utf-8",
    )
    print(f"saved IDs → {DPO_FILE_IDS_FILE}")


def create_dpo_job(args: argparse.Namespace) -> None:
    cfg = Config()
    client = get_openai_client()

    if args.train_file_id and args.val_file_id:
        train_id, val_id = args.train_file_id, args.val_file_id
    else:
        if not DPO_FILE_IDS_FILE.exists():
            raise RuntimeError("No uploaded DPO IDs found. Run upload-dpo or pass --train-file-id and --val-file-id.")
        ids = json.loads(DPO_FILE_IDS_FILE.read_text(encoding="utf-8"))
        train_id, val_id = ids["dpo_train_id"], ids["dpo_val_id"]

    print(f"Creating DPO job with train={train_id}, val={val_id}")
    job = client.fine_tuning.jobs.create(
        model=DPO_MODEL,
        training_file=train_id,
        validation_file=val_id,
        method={
            "type": "dpo",
            "dpo": {
                "hyperparameters": {
                    "n_epochs": cfg.dpo_n_epochs,
                    "batch_size": cfg.dpo_batch_size,
                    "learning_rate_multiplier": cfg.dpo_learning_rate_multiplier,
                }
            },
        },
        extra_body={"trainingType": "GlobalStandard"},
        suffix="haskell-format-dpo",
    )
    print(f"Job ID : {job.id}")
    print(f"Status : {job.status}")
    print(f"Model  : {job.model}")




def build_upload_create_dpo_job(args: argparse.Namespace) -> None:
    """One-shot command: run GHC filter, write DPO JSONLs, upload, create job."""
    build_args = argparse.Namespace(
        train_raw=args.train_raw,
        val_raw=args.val_raw,
        train_output=args.train_output,
        val_output=args.val_output,
        stats_output=args.stats_output,
        min_test_pass_rate=args.min_test_pass_rate,
        allow_compile_fail=args.allow_compile_fail,
    )
    build_dpo_jsonl(build_args)

    client = get_openai_client()
    train_id = upload_file(args.train_output, client)
    val_id = upload_file(args.val_output, client)
    DPO_FILE_IDS_FILE.parent.mkdir(parents=True, exist_ok=True)
    DPO_FILE_IDS_FILE.write_text(
        json.dumps({"dpo_train_id": train_id, "dpo_val_id": val_id}, indent=2),
        encoding="utf-8",
    )

    job_args = argparse.Namespace(train_file_id=train_id, val_file_id=val_id)
    create_dpo_job(job_args)



In [ ]:


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="DPO-only Haskell format fine-tuning pipeline")
    sub = parser.add_subparsers(dest="command", required=True)

    p_build = sub.add_parser("build-dpo-jsonl", help="Run GHC harness and build offline format-only DPO JSONL files from train_raw and val_raw")
    p_build.add_argument(
        "--train-raw",
        type=Path,
        default=DEFAULT_TRAIN_RAW_FILE,
        help="Input train_raw JSONL file. Default: train_raw.jsonl",
    )
    p_build.add_argument(
        "--val-raw",
        type=Path,
        default=DEFAULT_VAL_RAW_FILE,
        help="Input val_raw JSONL file. Default: val_raw.jsonl",
    )
    p_build.add_argument(
        "--train-output",
        type=Path,
        default=DPO_TRAIN_FILE,
        help="Output DPO training JSONL file.",
    )
    p_build.add_argument(
        "--val-output",
        type=Path,
        default=DPO_VAL_FILE,
        help="Output DPO validation JSONL file.",
    )
    p_build.add_argument(
        "--stats-output",
        type=Path,
        default=DPO_STATS_FILE,
        help="Output JSONL with per-row GHC/filter stats.",
    )
    p_build.add_argument(
        "--min-test-pass-rate",
        type=float,
        default=1.0,
        help="Minimum GHC test pass rate required to keep a DPO pair. Default: 1.0",
    )
    p_build.add_argument(
        "--allow-compile-fail",
        action="store_true",
        help="Do not require GHC compilation to keep a pair. Not recommended.",
    )
    p_build.set_defaults(func=build_dpo_jsonl)

    p_eval = sub.add_parser("evaluate-predictions", help="Evaluate marker formatting in a predictions JSONL")
    p_eval.add_argument("--predictions", required=True, type=Path)
    p_eval.add_argument("--output", type=Path, default=None, help="Optional JSONL with per-row format_eval attached")
    p_eval.set_defaults(func=evaluate_predictions)

    p_upload = sub.add_parser("upload-dpo", help="Upload DPO JSONL files and wait until imports are complete")
    p_upload.add_argument("--train-file", type=Path, default=DPO_TRAIN_FILE)
    p_upload.add_argument("--val-file", type=Path, default=DPO_VAL_FILE)
    p_upload.set_defaults(func=upload_dpo)

    p_job = sub.add_parser("create-dpo-job", help="Create Azure DPO fine-tuning job")
    p_job.add_argument("--train-file-id", default=None)
    p_job.add_argument("--val-file-id", default=None)
    p_job.set_defaults(func=create_dpo_job)

    p_all = sub.add_parser("build-upload-create-dpo-job", help="Run GHC harness, build DPO JSONLs, upload them, and create the DPO job")
    p_all.add_argument("--train-raw", type=Path, default=DEFAULT_TRAIN_RAW_FILE)
    p_all.add_argument("--val-raw", type=Path, default=DEFAULT_VAL_RAW_FILE)
    p_all.add_argument("--train-output", type=Path, default=DPO_TRAIN_FILE)
    p_all.add_argument("--val-output", type=Path, default=DPO_VAL_FILE)
    p_all.add_argument("--stats-output", type=Path, default=DPO_STATS_FILE)
    p_all.add_argument("--min-test-pass-rate", type=float, default=1.0)
    p_all.add_argument("--allow-compile-fail", action="store_true")
    p_all.set_defaults(func=build_upload_create_dpo_job)
    return parser


def main(argv: list[str] | None = None) -> None:
    parser = build_parser()
    args = parser.parse_args(argv)
    args.func(args)


def run_command(command: str) -> None:
    """Notebook-friendly command runner.

    Examples:
        run_command("build-dpo-jsonl --train-raw train_raw.jsonl --val-raw val_raw.jsonl")
        run_command("upload-dpo")
        run_command("create-dpo-job")
    """
    import shlex
    main(shlex.split(command))


def running_inside_notebook() -> bool:
    """True in Colab/Jupyter, where sys.argv contains a kernel JSON path."""
    argv_text = " ".join(map(str, sys.argv))
    return (
        "ipykernel" in sys.modules
        or "google.colab" in sys.modules
        or "kernel-" in argv_text and argv_text.endswith(".json")
    )


if __name__ == "__main__" and not running_inside_notebook():
    main()


# Get predictions and eval


In [7]:
run_command("build-upload-create-dpo-job --train-raw train_raw.jsonl --val-raw val_raw.jsonl")

Loading raw split files, not the full dataset...
train_raw: train_raw.jsonl (1600 rows)
val_raw  : val_raw.jsonl (200 rows)
Building tested, format-only DPO files from train_raw and val_raw...
No model generation is performed; no Azure deployment is needed for this step.
Each raw row is first run through the GHC/runghc harness; only passing rows become DPO pairs.


GHC+DPO val: 100%|██████████| 200/200 [01:06<00:00,  3.02it/s]


wrote  1033 rows → jsonl_outputs/dpo_training_format.jsonl
wrote   116 rows → jsonl_outputs/dpo_validation_format.jsonl
wrote  1800 rows → jsonl_outputs/dpo_generation_stats.jsonl

train GHC/filter summary
  raw rows:        1600
  compiled:        1060/1600
  full test pass:  1034/1600
  DPO pairs kept:  1033/1600
  skipped:
    ghc_compile_failed                                      540
    ghc_tests_below_threshold                               26
    preferred_output_does_not_satisfy_format_contract       1

val GHC/filter summary
  raw rows:        200
  compiled:        122/200
  full test pass:  116/200
  DPO pairs kept:  116/200
  skipped:
    ghc_compile_failed                                      78
    ghc_tests_below_threshold                               6

DPO JSONL build complete.
DPO train: jsonl_outputs/dpo_training_format.jsonl
DPO val  : jsonl_outputs/dpo_validation_format.jsonl
Stats    : jsonl_outputs/dpo_generation_stats.jsonl
Preference pairs: train=1033, val=11

In [31]:
def _format_component_flags(code: str) -> dict:
    lines = code.splitlines()
    nonempty = [(i, line.strip()) for i, line in enumerate(lines) if line.strip()]

    start_ok = bool(nonempty) and nonempty[0][1] == FORMAT_START
    end_ok = bool(nonempty) and nonempty[-1][1] == FORMAT_END

    sig_idx = first_top_level_type_signature_idx(code)
    signature_ok = (
        sig_idx is not None
        and sig_idx + 1 < len(lines)
        and lines[sig_idx + 1].strip() == FORMAT_SIGNATURE
    )

    return {
        "start_marker_ok": start_ok,
        "signature_marker_ok": signature_ok,
        "end_marker_ok": end_ok,
    }


def evaluate_test_predictions_with_ghc(
    predictions_path=TEST_PREDICTIONS_FILE,
    eval_output_path=TEST_EVAL_FILE,
    summary_output_path=TEST_SUMMARY_FILE,
):
    predictions_path = Path(predictions_path)
    eval_output_path = Path(eval_output_path)
    summary_output_path = Path(summary_output_path)

    if not predictions_path.exists():
        raise FileNotFoundError(f"Could not find predictions file: {predictions_path}")

    rows = load_jsonl(predictions_path)
    print(f"Loaded predictions: {len(rows)} from {predictions_path}")

    eval_rows = []
    iterator = tqdm(rows, desc="GHC evaluating predictions") if tqdm else rows

    for row in iterator:
        pred = row.get("prediction", "")
        tests = _normalize_tests_local(row.get("translated_test_cases"))

        try:
            ghc = run_tests(pred, tests)
        except Exception:
            ghc = {
                "compiled": False,
                "passed": 0,
                "total": len(tests) if tests else 1,
                "stdout": "",
                "stderr": traceback.format_exc(),
                "fmt": "harness_exception",
            }

        try:
            fmt_eval = score_format_constraints(pred)
        except Exception as e:
            fmt_eval = {
                "format_passed": 0,
                "format_total": 3,
                "format_score": 0.0,
                "format_errors": [f"format_eval_exception: {repr(e)}"],
            }

        component_flags = _format_component_flags(pred)

        total = max(int(ghc.get("total", 0)), 1)
        passed = int(ghc.get("passed", 0))
        compiled = bool(ghc.get("compiled", False))

        eval_rows.append({
            **row,
            "ghc_eval": ghc,
            "format_eval": fmt_eval,
            **component_flags,
            "compiled": compiled,
            "passed": passed,
            "total": total,
            "test_pass_rate": passed / total,
            "full_test_pass": compiled and passed == total,
            "perfect_format": (
                component_flags["start_marker_ok"]
                and component_flags["signature_marker_ok"]
                and component_flags["end_marker_ok"]
            ),
        })

    n = len(eval_rows)

    compiled_n = sum(1 for r in eval_rows if r["compiled"])
    full_pass_n = sum(1 for r in eval_rows if r["full_test_pass"])

    start_marker_n = sum(1 for r in eval_rows if r["start_marker_ok"])
    signature_marker_n = sum(1 for r in eval_rows if r["signature_marker_ok"])
    end_marker_n = sum(1 for r in eval_rows if r["end_marker_ok"])
    perfect_format_n = sum(1 for r in eval_rows if r["perfect_format"])

    avg_test_pass_rate = sum(r["test_pass_rate"] for r in eval_rows) / max(n, 1)

    total_tests = sum(r["total"] for r in eval_rows)
    total_passed = sum(r["passed"] for r in eval_rows)
    micro_test_pass_rate = total_passed / max(total_tests, 1)

    both_full_pass_and_format_n = sum(
        1 for r in eval_rows
        if r["full_test_pass"] and r["perfect_format"]
    )

    model_name = PREDICTION_MODEL_OR_DEPLOYMENT

    summary = {
        "model": model_name,
        "epochs": 1,
        "lr": 1.0,
        "batch_size": 8,

        "num_predictions": n,

        "start_marker": start_marker_n,
        "start_marker_rate": start_marker_n / max(n, 1),

        "signature_marker": signature_marker_n,
        "signature_marker_rate": signature_marker_n / max(n, 1),

        "end_marker": end_marker_n,
        "end_marker_rate": end_marker_n / max(n, 1),

        "compiled": compiled_n,
        "compiled_rate": compiled_n / max(n, 1),

        "functional_pass": full_pass_n,
        "functional_pass_rate": full_pass_n / max(n, 1),

        "avg_per_problem_test_pass_rate": avg_test_pass_rate,
        "micro_test_pass_rate": micro_test_pass_rate,
        "perfect_format": perfect_format_n,
        "perfect_format_rate": perfect_format_n / max(n, 1),
        "full_test_pass_and_perfect_format": both_full_pass_and_format_n,
        "full_test_pass_and_perfect_format_rate": both_full_pass_and_format_n / max(n, 1),
        "total_tests_passed": total_passed,
        "total_tests": total_tests,
    }

    write_jsonl(eval_rows, eval_output_path, bom=False)
    summary_output_path.parent.mkdir(parents=True, exist_ok=True)
    summary_output_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nTEST_RAW GHC + FORMAT SUMMARY")
    print(f"  model:                            {summary['model']}")
    print(f"  predictions:                      {n}")
    print(f"  start marker:                     {start_marker_n}/{n} ({100 * summary['start_marker_rate']:.2f}%)")
    print(f"  signature marker:                 {signature_marker_n}/{n} ({100 * summary['signature_marker_rate']:.2f}%)")
    print(f"  end marker:                       {end_marker_n}/{n} ({100 * summary['end_marker_rate']:.2f}%)")
    print(f"  compile rate:                     {compiled_n}/{n} ({100 * summary['compiled_rate']:.2f}%)")

    print("\nTABLE ROW")
    print("| Model | Epochs | LR | Batch Size | Start Marker | Signature Marker | End Marker | Compile Rate | Functional Pass Rate |")
    print("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    print(
        f"| {summary['model']} "
        f"| {summary['epochs']} "
        f"| {summary['lr']} "
        f"| {summary['batch_size']} "
        f"| {100 * summary['start_marker_rate']:.2f}% "
        f"| {100 * summary['signature_marker_rate']:.2f}% "
        f"| {100 * summary['end_marker_rate']:.2f}% "
        f"| {100 * summary['compiled_rate']:.2f}% "
    )

    print(f"\nWrote eval rows → {eval_output_path}")
    print(f"Wrote summary   → {summary_output_path}")

    return summary, eval_rows

In [30]:
summary, eval_rows = evaluate_test_predictions_with_ghc(
    predictions_path=TEST_PREDICTIONS_FILE,
    eval_output_path=TEST_EVAL_FILE,
    summary_output_path=TEST_SUMMARY_FILE,
)

summary

Loaded predictions: 200 from jsonl_outputs/test_predictions_dpo-nano.jsonl


GHC evaluating predictions: 100%|██████████| 200/200 [02:49<00:00,  1.18it/s]

wrote   200 rows → jsonl_outputs/test_predictions_dpo_ghc_eval-nano.jsonl

TEST_RAW GHC + FORMAT SUMMARY
  model:                            1-nano-2025-04-14-haskell-dpo
  predictions:                      200
  start marker:                     200/200 (100.00%)
  signature marker:                 148/200 (74.00%)
  end marker:                       199/200 (99.50%)
  compile rate:                     107/200 (53.50%)

TABLE ROW
| Model | Epochs | LR | Batch Size | Start Marker | Signature Marker | End Marker | Compile Rate | Functional Pass Rate |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 1-nano-2025-04-14-haskell-dpo | 1 | 1.0 | 8 | 100.00% | 74.00% | 99.50% | 53.50% 

Wrote eval rows → jsonl_outputs/test_predictions_dpo_ghc_eval-nano.jsonl
Wrote summary   → jsonl_outputs/test_predictions_dpo_ghc_summary-nano.json


{'model': '1-nano-2025-04-14-haskell-dpo',
 'epochs': 1,
 'lr': 1.0,
 'batch_size': 8,
 'num_predictions': 200,
 'start_marker': 200,
 'start_marker_rate': 1.0,
 'signature_marker': 148,
 'signature_marker_rate': 0.74,
 'end_marker': 199,
 'end_marker_rate': 0.995,
 'compiled': 107,
 'compiled_rate': 0.535,
 'functional_pass': 94,
 'functional_pass_rate': 0.47,
 'avg_per_problem_test_pass_rate': 0.5024215853805637,
 'micro_test_pass_rate': 0.6100848726909636,
 'perfect_format': 147,
 'perfect_format_rate': 0.735,
 'full_test_pass_and_perfect_format': 78,
 'full_test_pass_and_perfect_format_rate': 0.39,
 'total_tests_passed': 1222,
 'total_tests': 2003}